# Martin et al. (2023) acoustic aspiration-probe replication

Use a **T4 GPU** runtime. This evaluation only reads frozen acoustic checkpoints; it does not retrain or change Phase 1. Official MALD audio is about 1.5 GB and the formal nested-CV run is long. No Drive is used.

In [ ]:
# 1. Fetch the exact repository code without replacing Colab's PyTorch
import pathlib, subprocess, sys, importlib
PROJECT = pathlib.Path('/content/developmental_checkpoints_word_recognition')
if PROJECT.exists():
    subprocess.run(['git', '-C', str(PROJECT), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', 'https://github.com/ss-sebastian/developmental_checkpoints_word_recognition.git', str(PROJECT)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'requests>=2.31', 'scikit-learn>=1.3', 'matplotlib>=3.8'], check=True)
for name in list(sys.modules):
    if name == 'devlm' or name.startswith('devlm.'):
        del sys.modules[name]
sys.path.insert(0, str(PROJECT / 'src'))
importlib.invalidate_caches()
import devlm
print('devlm:', devlm.__file__)
print(subprocess.check_output(['git', '-C', str(PROJECT), 'rev-parse', '--short', 'HEAD'], text=True).strip())
import torch
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > T4 GPU, then reconnect.'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# 2. Upload exactly the acoustic Phase-1 output ZIP (M01--M30)
from google.colab import files
import zipfile, shutil
uploaded = files.upload()
if len(uploaded) != 1: raise ValueError('Upload exactly one acoustic-output ZIP')
archive = pathlib.Path(next(iter(uploaded)))
CHECKPOINT_ROOT = pathlib.Path('/content/acoustic_checkpoints')
if CHECKPOINT_ROOT.exists(): shutil.rmtree(CHECKPOINT_ROOT)
with zipfile.ZipFile(archive) as z:
    root = CHECKPOINT_ROOT.resolve()
    for info in z.infolist():
        if not (CHECKPOINT_ROOT / info.filename).resolve().is_relative_to(root): raise ValueError('Unsafe ZIP member')
    z.extractall(CHECKPOINT_ROOT)
from devlm.acoustic_aspiration_probe.run import discover_acoustic_checkpoints
checkpoints = discover_acoustic_checkpoints(CHECKPOINT_ROOT)
print('Validated', len(checkpoints), 'acoustic checkpoints:', [(i, round(h, 2)) for i, _, h in checkpoints])

In [ ]:
# 3. Formal run: cache PCA fits without changing the 10x10 nested-CV protocol.
# Set a positive value ONLY for a labelled small smoke test; leave None for the paper-analog formal analysis.
SMOKE_MAX_ITEMS = None
import numpy as np
import devlm.acoustic_aspiration_probe.run as probe_run
from devlm.acoustic_aspiration_probe.run import _resolve_fbank_backend, run_replication

# The run calls nested_cv_score repeatedly for PCA dimensions of the SAME X.
# Cache full PCA fits by (outer fold, inner fold) and slice their prefixes.
# Keeping the current X object strongly referenced prevents id reuse; a new X
# automatically clears every cached basis, bounding memory to one task at a time.
_PCA_CACHE = {'array': None, 'fits': {}}
_ORIGINAL_NESTED_CV = probe_run.nested_cv_score

def _cached_nested_cv_score(x, y, fold_by_id, ids, dimensions):
    if _PCA_CACHE['array'] is not x:
        _PCA_CACHE['array'] = x
        _PCA_CACHE['fits'].clear()
    folds = np.asarray([fold_by_id[item] for item in ids])
    classes = np.unique(y)
    if len(classes) < 2 or min(np.bincount(np.searchsorted(classes, y))) < 12:
        raise ValueError('Each class needs at least 12 items for outer-10/inner-10 CV')
    def project(train_indices, test_indices, key):
        width = min(dimensions, x.shape[1], len(train_indices) - 1)
        if width < 1:
            raise ValueError('Insufficient training samples for PCA')
        if width >= x.shape[1]:
            return x[train_indices], x[test_indices]
        if key not in _PCA_CACHE['fits']:
            # Full decomposition is exactly what _fit_pca already computes;
            # it is independent of requested prefix dimension and of C.
            _PCA_CACHE['fits'][key] = probe_run._fit_pca(x[train_indices], x.shape[1])
        mean, full_basis = _PCA_CACHE['fits'][key]
        basis = full_basis[:, :width]
        return (x[train_indices] - mean) @ basis, (x[test_indices] - mean) @ basis
    predictions = np.zeros((len(y), len(classes)), dtype=float)
    print(f'[optimized nested CV] d={dimensions}; outer 0/10', end='', flush=True)
    for outer in range(10):
        print(f'\r[optimized nested CV] d={dimensions}; outer {outer + 1}/10', end='', flush=True)
        outer_train = folds != outer
        outer_test = folds == outer
        outer_indices = np.flatnonzero(outer_train)
        inner_folds = probe_run.inner_ten_fold_assignment(ids, y, folds, outer)
        candidate_scores = []
        for c_value in probe_run.REGULARIZATION:
            inner_probability = np.zeros((len(outer_indices), len(classes)), dtype=float)
            for inner in range(10):
                train_indices = outer_indices[inner_folds != inner]
                test_indices = outer_indices[inner_folds == inner]
                train_x, test_x = project(train_indices, test_indices, ('inner', outer, inner))
                inner_probability[inner_folds == inner] = probe_run._fit_predict(train_x, y[train_indices], test_x, classes, c_value)
            candidate_scores.append((probe_run._score(y[outer_indices], inner_probability), c_value))
        chosen_c = max(candidate_scores)[1]  # same deterministic tie rule as source
        train_x, test_x = project(outer_indices, np.flatnonzero(outer_test), ('outer', outer))
        predictions[outer_test] = probe_run._fit_predict(train_x, y[outer_indices], test_x, classes, chosen_c)
    print()
    return probe_run._score(y, predictions)

# Equivalence check: same formal outer-10/fresh-inner-10/C/PCA semantics.
# It fits real three-class regressions before monkeypatching the formal runner.
_rng = np.random.default_rng(20261006)
_check_y = np.repeat(np.arange(3), 20)
_check_x = np.vstack([_rng.normal(loc=label * 2.0, scale=.25, size=(20, 4)) for label in range(3)])
_check_ids = [f'validation_{index}' for index in range(len(_check_y))]
_check_folds = probe_run.stratified_fixed_item_folds(_check_ids, _check_y)
for _dimension in (2, 4):
    _PCA_CACHE['array'] = None; _PCA_CACHE['fits'].clear()
    _reference = _ORIGINAL_NESTED_CV(_check_x, _check_y, _check_folds, _check_ids, _dimension)
    _optimized = _cached_nested_cv_score(_check_x, _check_y, _check_folds, _check_ids, _dimension)
    assert np.isclose(_reference, _optimized, rtol=1e-10, atol=1e-12), (_dimension, _reference, _optimized)
print('Validated PCA-cache equivalence at d=2 and full d=4.')
probe_run.nested_cv_score = _cached_nested_cv_score

_, FBANK_BACKEND = _resolve_fbank_backend(allow_compatibility_fallback=SMOKE_MAX_ITEMS is not None)
print('Verified baseline backend:', FBANK_BACKEND)
OUTPUT = pathlib.Path('/content/acoustic_aspiration_replication_outputs')
MALD_CACHE = pathlib.Path('/content/mald_official_cache')
run_replication(CHECKPOINT_ROOT, OUTPUT, MALD_CACHE, max_items=SMOKE_MAX_ITEMS)
print('Complete:', OUTPUT)

In [ ]:
# 4. Download all results, plots, fold assignments, and manifest
import shutil
archive = shutil.make_archive('/content/acoustic_aspiration_replication_outputs', 'zip', OUTPUT)
files.download(archive)